In [ ]:
# test code for working with data from jeol_data before putting it into pynxtools-em
import os
import re
from pathlib import Path

from pynxtools_em.examples.deu_berlin_koch.inspect_jeol_sidecar_variants import (
    JEOL_LAYOUT_ONE,
    JEOL_LAYOUT_TWO,
    STRING_DECODER_CODECS,
)
from pynxtools_em.examples.get_sha256_of_directories import SEPARATOR

print(os.getcwd())

In [ ]:
cnt = 0
layouts: dict[int, list[str]] = {1: JEOL_LAYOUT_ONE, 2: JEOL_LAYOUT_TWO}
for file in Path(f"{os.getcwd()}{os.sep}tests{os.sep}jeol_data{os.sep}txt").iterdir():
    if file.is_file():
        for codec in STRING_DECODER_CODECS:
            try:
                with open(file, encoding=codec) as fp:
                    txt = fp.readlines()
                    keys: list[int] = []
                    for key, layout in layouts.items():
                        conformant: bool = True
                        for idx, line in enumerate(txt):
                            if not re.fullmatch(layout[idx], line):
                                # print(
                                #     f"{key}, {idx}, {layout[idx]}, {SEPARATOR}{line}{SEPARATOR}"
                                # )
                                conformant = False
                                break
                        if conformant:
                            keys.append(key)
                            # print(f"{key}, layout conformant")

                    if len(keys) == 0:
                        print(f"{file}, {len(txt)}, {keys}")
                        for key, layout in layouts.items():
                            for idx, line in enumerate(txt):
                                if not re.fullmatch(layout[idx], line):
                                    print(
                                        f"{key}, {idx}, {layout[idx]}, {SEPARATOR}{line}{SEPARATOR}"
                                    )
                                    break
                    break
            except UnicodeDecodeError:
                continue
        cnt += 1
        # if cnt >= 100:
        #     break
print(cnt)
# another goody, JEOLs which date was this data taken? 5/4/2020 or 4/5/2020 well who cares if month and day is swopped but you need a start_time ?

In [ ]:
def does_file_conform_with_layout(path: str, layout: list[str]) -> bool:
    """Check if path is a text file and if so follows the specific line-by-line layout as defined in layout."""
    conforms: bool = True
    # if magic.from_file(path, mime=True) == "text/plain":  # libmagic alternative but outdated compared to
    if puremagic.from_file(path) == ".txt":
        with open(path) as fp:
            n_lines_layout: int = len(layout)
            txt = fp.readlines()
            for idx, line in enumerate(txt):
                if idx < n_lines_layout:
                    if not re.fullmatch(layout[idx], line):
                        conforms = False
                        break
                else:
                    conforms = False
                    break
    return conforms

In [ ]:
# as typical with JEOL, content is a text file,
# here check with how many variants of formatting we need to be dealing
# kaitai is a list of regex instructions that model the formatting of each expected line
print(does_file_conform_with_layout("tests/jeol_data/1.txt", layout_nerl))

In [ ]:
# test code for working with data from jeol_data before putting it into pynxtools-em
import re

import puremagic

from pynxtools_em.examples.get_sha256_of_directories import SEPARATOR

# test_path = "tests/jeol_data/1.txt"
# test_path = "tests/jeol_data/ZnGaO_ovw.txt"
test_path = "tests/jeol_data/ZnGaO_diff.txt"
# test_path = "tests/jeol_data/ZnGaO_stem01_ADF_CL10cm_spot07nm_25kx_ZA100_ovw.txt"
# print(puremagic.from_file(test_path))
# import magic
# print(magic.from_file(test_path, mime=True))
from charset_normalizer import from_bytes, from_path

result = from_path(test_path).best()
if result:
    print("encoding:", result.encoding)
    print("confidence:", result.percent_chaos)
best = from_bytes(open(test_path, "rb").read()).best()
if best:
    print(best.encoding)

STRING_DECODER_CODECS = [
    "utf-8",
    "cp1252",
    "utf-16",
    "utf-16-be",
    "utf-16-le",
    "latin-1",  # TODO not as robust ?
]

In [ ]:
BREAK = r"(?:\r\n?|\n)"
FLOAT = r"(?:\d+(?:\.\d*)?|\.\d+)"
INT = r"\d+"
DATE = r"([1-9]|0[1-9]|1[0-2])/(0[1-9]|[12][0-9]|3[0-1])/\d{4}"  # e.g. 5/25/2026
TIME = r"(?:0?[1-9]|1[0-2]):(?:[0-5][0-9]):(?:[0-5][0-9]) (AM|PM)"
CHARS_NO_BREAK = r"[^\r\n]*"  # + one or more, * zero or more, ? zero or one
LENGTH = r"\d+(?:\.\d+)?\s?(?:nm|µm)"

# JEOL, Hannah/20210225_CsPbBrI40Big_TEMIsrael/1.txt
layout_one: list[str] = [
    rf"^\$CM_FORMAT {BREAK}$",
    rf"^\$CM_VERSION {CHARS_NO_BREAK}{BREAK}$",
    rf"^\$CM_COMMENT  {BREAK}$",
    rf"^\$CM_DATE {DATE}{BREAK}$",
    rf"^\$CM_TIME {TIME}{BREAK}$",
    rf"^\$CM_OPERATOR {CHARS_NO_BREAK}{BREAK}$",
    rf"^\$CM_INSTRUMENT JEM-2200FS{BREAK}$",
    rf"^\$CM_NAME Specimen{BREAK}$",
    rf"^\$CM_FRAME_SIZE {INT} {INT}{BREAK}$",
    rf"^\$CM_DATA_BIT {INT}{BREAK}$",
    rf"^\$CM_EFECT_BIT {INT}{BREAK}$",
    rf"^\$CM_ACCEL_VOLT 200{BREAK}$",  # 200 to replace by {INT}
    rf"^\$CM_MAG {INT}{BREAK}$",
    rf"^\$CM_SIGNAL TEM{BREAK}$",
    rf"^\$\$EM_PIXELSPERMETER_X {FLOAT}{BREAK}$",
    rf"^\$\$EM_PIXELSPERMETER_Y {FLOAT}{BREAK}$",
]

# Robert/2021_03_19_ZnGaO/STEM/ZnGaO_stem01_ADF_CL10cm_spot07nm_25kx_ZA100_ovw.txt
layout_two: list[str] = [
    rf"^\$CM_FORMAT {BREAK}$",
    rf"^\$CM_VERSION 0.1{BREAK}$",
    rf"^\$CM_COMMENT {BREAK}$",
    rf"^\$CM_DATE {DATE}{BREAK}$",
    rf"^\$CM_TIME {TIME}{BREAK}$",
    rf"^\$CM_OPERATOR {CHARS_NO_BREAK}{BREAK}$",
    rf"^\$CM_INSTRUMENT JEM-2200FS{BREAK}$",
    rf"^\$CM_ACCEL_VOLT {FLOAT}{BREAK}$",
    rf"^\$CM_MAG {INT}{BREAK}$",
    rf"^\$CM_SIGNAL {CHARS_NO_BREAK}{BREAK}$",  # "DFI  " in the prototype
    rf"^\$\$SM_FILM_NUMBER {INT}{BREAK}$",
    rf"^\$\$SM_WD {FLOAT}{BREAK}$",
    rf"^\$\$SM_MICRON_BAR {INT}{BREAK}$",
    rf"^\$\$SM_MICRON_MARKER {LENGTH}{BREAK}$",
    rf"^\$\$SM_FONT_SIZE {INT} {INT}{BREAK}$",
    rf"^\$\$SM_DISPLAY_MODE {CHARS_NO_BREAK}{BREAK}$",
]

In [ ]:
raw = open(test_path, "rb").read()
# utf byte order mark
for enc, bom in [
    ("utf-8-sig", b"\xef\xbb\xbf"),
    ("utf-16-le", b"\xff\xfe"),
    ("utf-16-be", b"\xfe\xff"),
]:
    if raw.startswith(bom):
        txt = raw.decode(enc)
# utf-8
try:
    txt = raw.decode("utf-8")
except UnicodeDecodeError:
    pass

best = from_bytes(raw).best()
if best:
    txt = str(best)

with open(test_path, encoding="cp1252") as fp:
    txt = fp.readlines()

print(txt)

In [ ]:
# ! pip install charset-normalizer